# Replicación del análisis de aleatoriedad: Kino y juegos complementarios

**Extensión del análisis principal** — se aplica la misma metodología del notebook del Loto
sobre 4 bombos independientes del Kino (14 de 25), replicando los tests de uniformidad,
estadísticos de orden y corrección de Bonferroni.

---

| Bombo | Estructura | Sorteos |
|---|---|---|
| Kino principal | 14 de 25 | 889 |
| Chao Jefe 3M | 14 de 25 | 497 |
| Chao Jefe 2M | 14 de 25 | 370 |
| Super Combo Marraqueta | 14 de 25 | 370 |

Fuente: [chileresultados.com](https://www.chileresultados.com), extraídos por web scraping.

## 1. Carga y exploración de datos

Verificación de estructura, tipos, nulos y duplicados antes de analizar.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from math import comb
from pathlib import Path

# Configuración
N = 25   # bolillas en el bombo
K = 14   # bolillas extraídas

# Carga de datos
archivos = {
    "Kino principal": "kino_principal.csv",
    "Chao Jefe 3M": "kino_chao_jefe_3m.csv",
    "Chao Jefe 2M": "kino_chao_jefe_2m.csv",
    "Super Combo": "kino_super_combo.csv",
}

bombos = {}
for nombre, archivo in archivos.items():
    df = pd.read_csv(archivo, encoding="utf-8")
    cols = [c for c in df.columns if c.startswith("n")]
    for c in cols + ["sorteo"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=cols).reset_index(drop=True)
    for c in cols + ["sorteo"]:
        df[c] = df[c].astype(int)
    bombos[nombre] = {"df": df, "cols": cols}

print("="*65)
print(f"{'Bombo':<22} {'Filas':>6} {'Cols':>5} {'Nulos':>6} {'Dupl':>5} {'Rango OK':>9}")
print("-"*65)
for nombre, data in bombos.items():
    df = data["df"]; cols = data["cols"]
    nulos = df[cols].isnull().sum().sum()
    dupl = df.duplicated(subset=cols).sum()
    vals = df[cols].values.flatten()
    rango_ok = "✓" if (vals.min() >= 1 and vals.max() <= 25) else "✗"
    print(f"{nombre:<22} {len(df):>6} {len(cols):>5} {nulos:>6} {dupl:>5} {rango_ok:>9}")
print("="*65)

In [ ]:
# Verificar que cada sorteo tenga 14 números distintos
print("Validación de unicidad (14 números distintos por sorteo):")
for nombre, data in bombos.items():
    df = data["df"]; cols = data["cols"]
    unicos = df[cols].apply(lambda r: len(set(r)), axis=1)
    problemas = (unicos != K).sum()
    print(f"  {nombre}: {problemas} sorteos con números repetidos" if problemas > 0
          else f"  {nombre}: ✓ todos con 14 números distintos")

## 2. Test de uniformidad — Chi-cuadrado por bombo

Misma lógica que en el análisis del Loto: si los bombos son justos, cada número
del 1 al 25 debe salir con frecuencia cercana a la esperada.

In [ ]:
print(f"{'Bombo':<22} {'Chi²':>9} {'p-value':>10}  {'Caliente':<10} {'Veredicto'}")
print("-"*70)
for nombre, data in bombos.items():
    df = data["df"]; cols = data["cols"]
    todos = pd.concat([df[c] for c in cols])
    freq = np.array([np.sum(todos == i) for i in range(1, N+1)])
    esperada = len(df) * K / N
    chi2, p = stats.chisquare(freq, f_exp=[esperada]*N)
    cal = np.argmax(freq) + 1
    print(f"{nombre:<22} {chi2:>9.2f} {p:>10.4f}   #{cal:<9} {'uniforme ✓' if p > 0.05 else 'SESGO'}")

Los 4 bombos pasan el test de uniformidad individualmente. Cada uno tiene
un número caliente distinto — la firma del azar, no del sesgo.

## 3. Replicación con corrección de Bonferroni

Con 4 tests simultáneos, el umbral ajustado es α = 0.05/4 = 0.0125.
Es el mismo diseño de replicación que se usó con los 4 bombos del Loto.

In [ ]:
alpha_bonf = 0.05 / len(bombos)
print(f"Umbral Bonferroni: α = {alpha_bonf}\n")

print(f"{'Bombo':<22} {'p-value':>10}  {'Veredicto (Bonferroni)'}")
print("-"*55)
rechazos = 0
for nombre, data in bombos.items():
    df = data["df"]; cols = data["cols"]
    todos = pd.concat([df[c] for c in cols])
    freq = np.array([np.sum(todos == i) for i in range(1, N+1)])
    esperada = len(df) * K / N
    _, p = stats.chisquare(freq, f_exp=[esperada]*N)
    veredicto = "SESGO" if p < alpha_bonf else "uniforme ✓"
    if p < alpha_bonf: rechazos += 1
    print(f"{nombre:<22} {p:>10.4f}   {veredicto}")

print(f"\nResultado: {rechazos} de {len(bombos)} bombos rechazan H0 tras Bonferroni.")
if rechazos == 0:
    print("Ningún bombo muestra sesgo. Replicación exitosa.")

## 4. Valores extremos — ¿El número caliente es sospechoso?

Misma simulación Monte Carlo que en el Loto: con 25 números,
alguno siempre queda arriba por azar. Se cuantifica si el caliente
observado cae dentro del rango esperado.

In [ ]:
np.random.seed(1)

for nombre, data in bombos.items():
    df = data["df"]; cols = data["cols"]
    todos = pd.concat([df[c] for c in cols])
    freq = np.array([np.sum(todos == i) for i in range(1, N+1)])
    esperada = len(df) * K / N
    std = np.sqrt(len(df) * K * (1/N) * (1 - 1/N))
    desvio_real = (freq.max() - esperada) / std

    # Simulación
    maximos = []
    for _ in range(3000):
        sim = np.random.multinomial(len(df) * K, [1/N]*N)
        maximos.append((sim.max() - esperada) / std)
    maximos = np.array(maximos)

    print(f"{nombre}:")
    print(f"  Caliente #{np.argmax(freq)+1} a {desvio_real:.2f}σ")
    print(f"  Rango esperado por azar (5-95%): {np.percentile(maximos,5):.2f}σ a {np.percentile(maximos,95):.2f}σ")
    dentro = "✓ dentro del rango" if np.percentile(maximos,5) <= desvio_real <= np.percentile(maximos,95) else "⚠ fuera del rango"
    print(f"  {dentro}\n")

## 5. Estadísticos de orden — estructura posicional

Con 14 posiciones ordenadas de 25, cada posición tiene una media teórica exacta:
E[X(k)] = k(N+1)/(K+1). Se compara la distribución observada contra la predicción
combinatoria pura.

In [ ]:
def prob_orden(k, x, N_bombo=25, K_ext=14):
    if x < k or x > N_bombo - (K_ext - k):
        return 0
    return comb(x-1, k-1) * comb(N_bombo-x, K_ext-k) / comb(N_bombo, K_ext)

# Usar el Kino principal (más datos)
df_kp = bombos["Kino principal"]["df"]
cols_kp = bombos["Kino principal"]["cols"]

print(f"Estadísticos de orden — Kino principal ({len(df_kp)} sorteos)")
print(f"{'Pos':<5} {'Teórica':>9} {'Observada':>10} {'Corr':>7}")
print("-"*35)

correlaciones = []
for k in range(1, K+1):
    teo = k * (N+1) / (K+1)
    col = df_kp[cols_kp[k-1]].values
    obs = col.mean()
    o = np.array([np.sum(col == v) for v in range(1, N+1)], dtype=float)
    if o.sum() > 0: o /= o.sum()
    t = np.array([prob_orden(k, x) for x in range(1, N+1)])
    if o.std() > 0 and t.std() > 0:
        r = np.corrcoef(o, t)[0, 1]
        correlaciones.append(r)
    else:
        r = float('nan')
    print(f" n{k:<3} {teo:>9.2f} {obs:>10.2f} {r:>7.4f}")

print(f"\nCorrelación media: {np.mean(correlaciones):.4f}")
print("(>0.99 indica ajuste perfecto a la teoría combinatoria)")

Las medias observadas coinciden con las teóricas y las correlaciones superan 0.99.
La estructura posicional es una propiedad de la combinatoria, no de los datos:
se calcula sin ver un solo sorteo real. Los datos solo la confirman.

## 6. Paridad, suma y memoria entre sorteos

In [ ]:
df_kp = bombos["Kino principal"]["df"]
cols_kp = bombos["Kino principal"]["cols"]

# Pares por sorteo
pares = df_kp[cols_kp].apply(lambda r: sum(1 for x in r if x % 2 == 0), axis=1)
print(f"Pares por sorteo: {pares.mean():.2f} (teórico {K * 12/N:.2f})")

# Suma
sumas = df_kp[cols_kp].sum(axis=1)
print(f"Suma media: {sumas.mean():.1f} (teórico {K * (N+1)/2:.1f})")
print(f"Rango 80% central: [{sumas.quantile(.1):.0f}, {sumas.quantile(.9):.0f}]")

# Memoria entre sorteos consecutivos
ls = df_kp.sort_values("sorteo").reset_index(drop=True)
reps = []
for i in range(1, len(ls)):
    reps.append(len(set(ls.loc[i-1, cols_kp]) & set(ls.loc[i, cols_kp])))
print(f"\nRepeticiones entre sorteos consecutivos: {np.mean(reps):.2f} (teórico {K*K/N:.2f})")
print("Sin memoria entre sorteos." if abs(np.mean(reps) - K*K/N) < 0.5 else "⚠ Diferencia notable.")

## 7. Comparación cruzada — ¿Los bombos comparten calientes?

Si hubiera un defecto mecánico compartido, el mismo número saldría caliente
en todos los bombos. Si es azar, cada bombo tiene su propio caliente.

In [ ]:
print(f"{'Bombo':<22} {'Caliente':>9} {'Frío':>6}")
print("-"*40)
calientes = []
for nombre, data in bombos.items():
    df = data["df"]; cols = data["cols"]
    todos = pd.concat([df[c] for c in cols])
    freq = np.array([np.sum(todos == i) for i in range(1, N+1)])
    cal = np.argmax(freq) + 1
    frio = np.argmin(freq) + 1
    calientes.append(cal)
    print(f"{nombre:<22} #{cal:>7} #{frio:>4}")

print(f"\nNúmeros calientes únicos: {len(set(calientes))} de {len(calientes)} bombos")
if len(set(calientes)) == len(calientes):
    print("Todos distintos — firma del azar, no del defecto.")
else:
    print("Algunos coinciden — pero puede ser azar con solo 25 opciones posibles.")

## 8. Conclusiones

| Análisis | Resultado |
|---|---|
| Uniformidad (χ², 4 bombos) | Todos uniformes |
| Bonferroni (α = 0.0125) | Ningún rechazo |
| Valores extremos | Dentro del rango esperado |
| Estadísticos de orden | r > 0.99 vs teoría |
| Paridad / suma / memoria | Coinciden con teórico |
| Calientes cruzados | Distintos entre bombos |

**La replicación Bonferroni sobre 4 bombos del Kino confirma el mismo resultado**
**que los 4 bombos del Loto:** no hay sesgo detectable. Dos loterías con estructura
combinatoria opuesta (6/41 vs 14/25), 8 bombos independientes en total, misma conclusión.

---

*Notebook complementario al análisis principal (`analisis_loteria_completo.ipynb`).*
*Pendiente: incorporar ReKino, RequeteKino y Premios Especiales cuando se depure el scraper.*